# Deep Ensembles — Simple and Scalable Predictive Uncertainty Estimation using Deep Ensembles (2017)

_Papers · Meta-learning, Bayesian & Robustness_

**Train a handful of probabilistic networks from different random starts and average them — you get honest, well-calibrated uncertainty, no Bayesian machinery.**

---

This notebook is a practice scaffold for the **Deep Ensembles — Simple and Scalable Predictive Uncertainty Estimation using Deep Ensembles (2017)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import math, numpy as np

# --- 0. Sanity-check the worked examples. ---
# (A) Gaussian NLL for one prediction: mu=2.0, var=0.5, y=3.0 (Eqn. 1, drop constant).
mu, var, y = 2.0, 0.5, 3.0
nll = 0.5*math.log(var) + (y-mu)**2/(2*var)
print("worked (A): 0.5*log(0.5)=%.6f  (3-2)^2/(2*0.5)=%.6f  NLL=%.6f" % (
      0.5*math.log(var), (y-mu)**2/(2*var), nll))
# worked (A): 0.5*log(0.5)=-0.346574  (3-2)^2/(2*0.5)=1.000000  NLL=0.653426

# (B) Combine two members (Sec 2.4): m1=(2.0,0.5), m2=(4.0,1.0), M=2.
mus, vrs, M = [2.0, 4.0], [0.5, 1.0], 2
mu_star = sum(mus)/M
var_star = sum(v + m*m for v, m in zip(vrs, mus))/M - mu_star**2
print("worked (B): mu*=%.2f  var*=%.2f" % (mu_star, var_star))
# worked (B): mu*=3.00  var*=1.75


# --- 1. The probabilistic model: 1 -> 50 -> 50 -> 2 outputs (mean, variance). ---
def make_net():
    return nn.Sequential(nn.Linear(1, 50), nn.ReLU(),
                         nn.Linear(50, 50), nn.ReLU(),
                         nn.Linear(50, 2))         # 2 outputs: mean, raw-variance

def split(out):
    mean = out[:, :1]
    variance = F.softplus(out[:, 1:]) + 1e-6       # positivity (Sec 2.2.1)
    return mean, variance

def gaussian_nll(mean, variance, target):          # Eqn. 1, mean over batch
    return (0.5*torch.log(variance) + (target-mean)**2/(2*variance)).mean()

# --- 2. Training data: inputs ONLY in [-3, 3]; uncertainty should grow OUTSIDE. ---
def truef(x): return np.sin(x) + 0.1*x
np.random.seed(0)
xtr = np.random.uniform(-3, 3, (80, 1)).astype(np.float32)
ytr = (truef(xtr) + np.random.normal(0, 0.1, (80, 1))).astype(np.float32)
Xtr, Ytr = torch.tensor(xtr), torch.tensor(ytr)

# --- 3. Train ONE member, from its own seed, with optional adversarial term (Sec 2.3). ---
def train_member(seed, adversarial=True, eps=0.08):
    torch.manual_seed(seed); np.random.seed(seed)
    net = make_net(); opt = torch.optim.Adam(net.parameters(), lr=0.01)
    for _ in range(300):
        opt.zero_grad()
        X = Xtr.clone().requires_grad_(True)
        m, v = split(net(X)); loss = gaussian_nll(m, v, Ytr)
        if adversarial:                            # fast gradient sign method
            g = torch.autograd.grad(loss, X, retain_graph=True)[0]
            Xadv = (X + eps*torch.sign(g)).detach()
            ma, va = split(net(Xadv)); loss = loss + gaussian_nll(ma, va, Ytr)
        loss.backward(); opt.step()
    return net

# --- 4. The ENSEMBLE: M=5 members from DIFFERENT seeds (the only source of diversity). ---
M = 5
members = [train_member(s) for s in range(M)]
single  = members[0]                               # the M=1 ablation baseline

# --- 5. Combine via the Sec 2.4 mixture formulas. ---
def predict(nets, X):
    ms, vs = [], []
    with torch.no_grad():
        for n in nets:
            m, v = split(n(X)); ms.append(m); vs.append(v)
    ms, vs = torch.stack(ms), torch.stack(vs)
    mu_s = ms.mean(0)
    var_s = (vs + ms**2).mean(0) - mu_s**2         # avg var + disagreement of means
    return mu_s.squeeze(1).numpy(), var_s.squeeze(1).numpy()

# --- 6. Compare ensemble vs single net far OUTSIDE the training band. ---
xte = np.linspace(-7, 7, 57).astype(np.float32).reshape(-1, 1)
Xte = torch.tensor(xte); xg = xte.squeeze(1)
_, var_e = predict(members, Xte); std_e = np.sqrt(var_e)
_, var_s = predict([single],  Xte); std_s = np.sqrt(var_s)
inside  = np.abs(xg) <= 3.0
outside = np.abs(xg) >= 5.0
print("\nMean predicted std  (training inputs only in [-3,3]):")
print("  ENSEMBLE (M=5): inside=%.3f  outside=%.3f  ratio=%.2f" % (
      std_e[inside].mean(), std_e[outside].mean(), std_e[outside].mean()/std_e[inside].mean()))
print("  SINGLE   (M=1): inside=%.3f  outside=%.3f  ratio=%.2f" % (
      std_s[inside].mean(), std_s[outside].mean(), std_s[outside].mean()/std_s[inside].mean()))
# ENSEMBLE (M=5): inside=0.116  outside=0.219  ratio=1.89   <- uncertainty GROWS away from data
# SINGLE   (M=1): inside=0.112  outside=0.059  ratio=0.53   <- single net SHRINKS: overconfident OOD
# (Our small run, not the paper's reported numbers. Exact values vary by seed/hardware.)

## Visualize the data & results

_As inputs move AWAY from the training band ([-3,3]), how does each method's predicted uncertainty behave — a 5-member deep ensemble vs a single probabilistic network?_

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, math, numpy as np

def make_net(): return nn.Sequential(nn.Linear(1,50), nn.ReLU(),
                                     nn.Linear(50,50), nn.ReLU(), nn.Linear(50,2))
def split(o):
    m = o[:, :1]; v = F.softplus(o[:, 1:]) + 1e-6; return m, v
def gnll(m, v, y): return (0.5*torch.log(v) + (y-m)**2/(2*v)).mean()
def truef(x): return np.sin(x) + 0.1*x

np.random.seed(0)
xtr = np.random.uniform(-3,3,(80,1)).astype(np.float32)
ytr = (truef(xtr)+np.random.normal(0,0.1,(80,1))).astype(np.float32)
Xtr, Ytr = torch.tensor(xtr), torch.tensor(ytr)

def train_member(seed, eps=0.08):                    # different seed per member
    torch.manual_seed(seed); np.random.seed(seed)
    net = make_net(); opt = torch.optim.Adam(net.parameters(), lr=0.01)
    for _ in range(300):
        opt.zero_grad(); X = Xtr.clone().requires_grad_(True)
        m, v = split(net(X)); loss = gnll(m, v, Ytr)
        g = torch.autograd.grad(loss, X, retain_graph=True)[0]   # FGSM (Sec 2.3)
        Xa = (X + eps*torch.sign(g)).detach()
        ma, va = split(net(Xa)); (loss + gnll(ma, va, Ytr)).backward(); opt.step()
    return net

members = [train_member(s) for s in range(5)]        # M=5
single  = members[0]

def predict(nets, X):
    ms, vs = [], []
    with torch.no_grad():
        for n in nets:
            m, v = split(n(X)); ms.append(m); vs.append(v)
    ms, vs = torch.stack(ms), torch.stack(vs)
    mu = ms.mean(0); var = (vs + ms**2).mean(0) - mu**2   # Sec 2.4 combination
    return np.sqrt(var.squeeze(1).numpy())

xg = np.linspace(-7,7,15).astype(np.float32).reshape(-1,1)
std_e = predict(members, torch.tensor(xg))
std_s = predict([single], torch.tensor(xg))
print("x        :", [round(float(v),0) for v in xg.squeeze(1)])
print("ensemble :", [round(float(v),3) for v in std_e])
print("single   :", [round(float(v),3) for v in std_s])
# ensemble : [0.164, 0.168, 0.129, 0.104, 0.102, 0.112, 0.141, 0.124, 0.086,
#             0.115, 0.123, 0.159, 0.206, 0.279, 0.357]   <- grows toward x=7
# single   : [0.033, 0.046, 0.065, 0.087, 0.101, 0.105, 0.136, 0.122, 0.090,
#             0.109, 0.117, 0.103, 0.088, 0.071, 0.057]   <- shrinks away from data
# Our small run, not the paper's number.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The single-net ablation. You have a working ensemble of $M = 5$. Set $M = 1$ (use one member as
            the whole predictor) and look at the predicted standard deviation far outside the training band. Which
            term of the &sect;2.4 variance did you just zero out, and what do you expect to happen to the
            out-of-distribution uncertainty?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall the two parts of $\sigma_*^2$: the average member variance, and the variance of the member means (the disagreement). — _With $M=1$ there is only one mean, so the disagreement term $\tfrac{1}{M}\sum_m\mu_m^2 - \mu_*^2$ is exactly zero._
- Conclude what is left: only that single member's own predicted variance $\sigma_1^2(x)$ — a function it fit on the training band. — _Outside the band the variance head was never trained, so it can shrink, leaving the prediction overconfident where it should be unsure._
- Run it and compare the std at $x=7$ for $M=1$ versus $M=5$. — _The ensemble's std climbs away from the data; the single net's does not — the disagreement term is what was doing the work._

**Answer:** You zeroed out the disagreement term — the variance of the member means. With $M=1$
                 the ensemble variance collapses to the lone member's own $\sigma_1^2(x)$, which is just a fitted
                 function with no obligation to grow outside the training band. In our run the single net's
                 predicted std actually shrinks away from the data (it grows overconfident out-of-distribution),
                 while the ensemble's std rises sharply. The growth of uncertainty far from the data comes
                 almost entirely from members disagreeing — the thing a single network cannot do.

</details>

**Problem 2.** In the worked example, member 1 predicted $\mu_1=2.0,\ \sigma_1^2=0.5$ and member 2 predicted
            $\mu_2=4.0,\ \sigma_2^2=1.0$, giving ensemble variance $\sigma_*^2 = 1.75$. Now suppose the two
            members agree: both predict $\mu = 3.0$ (same variances). Recompute $\sigma_*^2$. What does
            the change tell you?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Ensemble mean: $\mu_* = \tfrac{1}{2}(3.0+3.0)=3.0$ — unchanged. — _Both means are now 3.0._
- Average of $(\sigma_m^2+\mu_m^2)$: $\tfrac{1}{2}[(0.5+9)+(1.0+9)] = \tfrac{1}{2}(9.5+10.0)=9.75$. — _Each $\mu_m^2 = 3.0^2 = 9$ now._
- Ensemble variance: $9.75 - 3.0^2 = 9.75 - 9.0 = 0.75$. — _The disagreement term vanished; only the average member variance $\tfrac{1}{2}(0.5+1.0)=0.75$ remains._

**Answer:** $\sigma_*^2 = 0.75$, down from $1.75$. When the members agree, the disagreement term is exactly
                 zero and the ensemble variance is just the average of the members' own variances ($0.75$). The
                 missing $1.0$ in the original example was pure disagreement. This is the mechanism in miniature:
                 agreement $\Rightarrow$ low ensemble uncertainty; disagreement $\Rightarrow$ high — which is
                 precisely what you want as inputs drift away from the training data.

</details>

**Problem 3.** Why does the Gaussian negative-log-likelihood loss keep the network from "cheating" on a hard example
            by predicting an enormous variance, and how does that differ from plain mean-squared error?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write the loss for one example: $\tfrac{1}{2}\log\sigma^2 + \frac{(y-\mu)^2}{2\sigma^2}$. — _Two competing terms: a data-fit term divided by variance, and a penalty for large variance._
- Imagine sending $\sigma^2 \to \infty$ to kill the data-fit term. — _The second term $\frac{(y-\mu)^2}{2\sigma^2}\to 0$, but the first term $\tfrac{1}{2}\log\sigma^2\to+\infty$ — the penalty dominates, so a huge variance is bad too._
- Compare with mean-squared error, which is only $(y-\mu)^2$ with $\sigma^2$ fixed. — _Plain MSE has no variance to tune and no $\log\sigma^2$ term, so it cannot express or be penalized for confidence._

**Answer:** The $\tfrac{1}{2}\log\sigma^2$ term charges the network for claiming a large variance, so it
                 cannot escape a hard point by declaring itself maximally unsure — doing so sends the loss to
                 $+\infty$. The balance between "fit the target" (second term) and "don't over-claim variance"
                 (first term) is what produces honest, calibrated uncertainty. Plain mean-squared error is
                 the special case where the variance is fixed to a constant: only the $(y-\mu)^2$ term survives,
                 and the model has no way to say how sure it is. The Gaussian NLL strictly generalizes MSE by
                 learning the variance too.

</details>